### 3 · Retrieval Strategies — Comparing Approaches

This notebook loads the persisted FAISS index and experiments with different
retrieval strategies to find the best fit for our confluence docs.

### Strategies tested
```
1. Baseline         → vanilla similarity search (k=5)
2. Metadata filter   → similarity + doc_type / section filter
3. Multi-query       → LLM generates query variants, merges results
4. Comparison        → side-by-side evaluation on test questions
```

In [1]:
import logging
from pathlib import Path
from pprint import pprint

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

# Load the persisted FAISS index
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)
print(f"Loaded FAISS index: {vectorstore.index.ntotal} vectors")

Loaded FAISS index: 292 vectors


In [2]:
# ── Helper: display results consistently ─────────────────────────────────────

def show_results(results, label="Results"):
    """Print retrieval results in a consistent format."""
    print(f"\n{label} ({len(results)} chunks)")
    print("-" * 80)
    for i, doc in enumerate(results):
        m = doc.metadata
        print(f"  [{i}] {m.get('source', '?'):<45} section: {m.get('section', '?')[:45]}")
        print(f"      type={m.get('doc_type', '?'):<15} chunk={m.get('chunk_index', '?')}")
        print(f"      {doc.page_content[:120]}...")
        print()


# Test questions we'll use across all strategies
TEST_QUERIES = [
    "How is state management handled in the application?",
    "What are the code quality standards?",
    "How does the journey canvas work?",
    "What is the deployment process?",
]

---
### Strategy 1 — Baseline Similarity Search

Vanilla top-k nearest neighbor search. No filtering, no query expansion.
This is our **baseline** to compare against.

In [3]:
baseline_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

query = TEST_QUERIES[0]
print(f"Query: {query}")
results = baseline_retriever.invoke(query)
show_results(results, "Baseline")

Query: How is state management handled in the application?

Baseline (5 chunks)
--------------------------------------------------------------------------------
  [0] architecture/overview.md                      section: When to Use What
      type=architecture    chunk=13
      ## State Management Strategy  
### When to Use What  
| State Type | Technology | Examples |
|---|---|---|
| **Server da...

  [1] architecture/code-quality.md                  section: What Goes Where
      type=architecture    chunk=11
      ## State Management Rules  
### What Goes Where  
| State Type | Technology | Example |
|---|---|---|
| Server data (jou...

  [2] architecture/redux.md                         section: Overview
      type=architecture    chunk=1
      ## Overview  
The AMS Admin Tool uses **Redux Toolkit** for global client-side state management. Redux handles state tha...

  [3] features/journey-canvas.md                    section: Builder Slice (`builderSlice`)
      type=feature    

---
### Strategy 2 — Metadata Filtering

FAISS supports filtering by metadata fields **after** similarity search.
This lets you scope retrieval to a specific `doc_type`, `audience`, or `section`.

Useful when:
- The agent knows what category of docs to search
- You want to restrict answers to a specific document type

In [4]:
query = TEST_QUERIES[0]
print(f"Query: {query}\n")

# 2a: Filter by doc_type
results_arch = vectorstore.similarity_search(
    query, k=5, filter={"doc_type": "architecture"}
)
show_results(results_arch, "Filtered: doc_type='architecture'")

# 2b: Filter by doc_type = feature
results_feat = vectorstore.similarity_search(
    query, k=5, filter={"doc_type": "feature"}
)
show_results(results_feat, "Filtered: doc_type='feature'")

Query: How is state management handled in the application?


Filtered: doc_type='architecture' (5 chunks)
--------------------------------------------------------------------------------
  [0] architecture/overview.md                      section: When to Use What
      type=architecture    chunk=13
      ## State Management Strategy  
### When to Use What  
| State Type | Technology | Examples |
|---|---|---|
| **Server da...

  [1] architecture/code-quality.md                  section: What Goes Where
      type=architecture    chunk=11
      ## State Management Rules  
### What Goes Where  
| State Type | Technology | Example |
|---|---|---|
| Server data (jou...

  [2] architecture/redux.md                         section: Overview
      type=architecture    chunk=1
      ## Overview  
The AMS Admin Tool uses **Redux Toolkit** for global client-side state management. Redux handles state tha...

  [3] architecture/overview.md                      section: Overview
      type=archite

In [5]:
# 2c: Combine multiple filters — search within a specific document

query = "What are the anti-patterns to avoid?"
print(f"Query: {query}\n")

results_specific = vectorstore.similarity_search(
    query, k=5, filter={"source": "architecture/code-quality.md"}
)
show_results(results_specific, "Filtered: source='architecture/code-quality.md'")

Query: What are the anti-patterns to avoid?


Filtered: source='architecture/code-quality.md' (5 chunks)
--------------------------------------------------------------------------------
  [0] architecture/code-quality.md                  section: Anti-Patterns
      type=architecture    chunk=12
      ### Anti-Patterns  
- **Never** store server data in Redux -- use TanStack Query
- **Never** dispatch in render -- use `...

  [1] architecture/code-quality.md                  section: Performance Guidelines
      type=architecture    chunk=27
      ## Performance Guidelines  
- **Memoize** components with `memo()` -- especially list items rendered in loops
- **useCal...

  [2] architecture/code-quality.md                  section: Test Pattern (AAA)
      type=architecture    chunk=18
      ### Test Pattern (AAA)  
```typescript
describe('useComponentName', () => {
it('returns correct initial state', () => {
...

  [3] architecture/code-quality.md                  section: Overview
    

---
### Strategy 3 — Multi-Query Retriever

Uses an LLM to generate **multiple rephrasings** of the user's question,
runs each variant against the vectorstore, and merges + deduplicates the results.

This improves **recall** — different phrasings can surface chunks that a single
query misses. Tradeoff: 1 extra LLM call + 3-5× embedding lookups per query.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
)

# Enable logging to see the generated query variants
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.DEBUG)

query = TEST_QUERIES[0]
print(f"Original query: {query}\n")
results = multi_query_retriever.invoke(query)
show_results(results, "Multi-Query")

Original query: How is state management handled in the application?


Multi-Query (6 chunks)
--------------------------------------------------------------------------------
  [0] architecture/overview.md                      section: When to Use What
      type=architecture    chunk=13
      ## State Management Strategy  
### When to Use What  
| State Type | Technology | Examples |
|---|---|---|
| **Server da...

  [1] architecture/code-quality.md                  section: What Goes Where
      type=architecture    chunk=11
      ## State Management Rules  
### What Goes Where  
| State Type | Technology | Example |
|---|---|---|
| Server data (jou...

  [2] architecture/redux.md                         section: Overview
      type=architecture    chunk=1
      ## Overview  
The AMS Admin Tool uses **Redux Toolkit** for global client-side state management. Redux handles state tha...

  [3] architecture/overview.md                      section: Cross-Cutting Concerns
      type=archit

In [8]:
# Test with a more ambiguous query where multi-query should help

query = "How do I make sure my code is good?"
print(f"Original query: {query}\n")

print("=" * 80)
print("BASELINE")
print("=" * 80)
baseline_results = baseline_retriever.invoke(query)
show_results(baseline_results, "Baseline")

print("\n" + "=" * 80)
print("MULTI-QUERY")
print("=" * 80)
mq_results = multi_query_retriever.invoke(query)
show_results(mq_results, "Multi-Query")

Original query: How do I make sure my code is good?

BASELINE

Baseline (5 chunks)
--------------------------------------------------------------------------------
  [0] architecture/code-quality.md                  section: Code Review Checklist
      type=architecture    chunk=23
      ## Code Review Checklist  
Before submitting code, verify:  
```
[ ] Correct folder (atoms/molecules/organisms/pages)
[ ...

  [1] architecture/code-quality.md                  section: Code Quality & Style Guide for Developers
      type=architecture    chunk=0
      # Code Quality & Style Guide for Developers  
> **Parent:** [Architecture](./architecture.md)
> **Last Updated:** March ...

  [2] operations/code-quality-audit-plan.md         section: Code Quality & Audit Readiness Plan
      type=operations      chunk=0
      # Code Quality & Audit Readiness Plan  
**Date:** March 18, 2026
**Project:** AMS Admin Tool
**Purpose:** Prioritized pl...

  [3] architecture/code-quality.md                  se

---
### Strategy 4 — Side-by-Side Comparison

Run all test queries through baseline and multi-query retrievers.
Compare the **unique sources** retrieved — more unique sources = better recall.

In [9]:
# Turn off verbose logging for clean comparison output
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

print(f"{'Query':<55} {'Baseline':>10} {'Multi-Q':>10}")
print("-" * 80)

for query in TEST_QUERIES:
    baseline_docs = baseline_retriever.invoke(query)
    mq_docs = multi_query_retriever.invoke(query)

    baseline_sources = set(d.metadata["source"] for d in baseline_docs)
    mq_sources = set(d.metadata["source"] for d in mq_docs)

    print(f"{query[:53]:<55} {len(baseline_docs):>5}/{len(baseline_sources):>2}u  {len(mq_docs):>5}/{len(mq_sources):>2}u")

print("\n(format: total_chunks / unique_sources)")

Query                                                     Baseline    Multi-Q
--------------------------------------------------------------------------------
How is state management handled in the application?         5/ 4u      6/ 4u
What are the code quality standards?                        5/ 2u      8/ 4u
How does the journey canvas work?                           5/ 2u      7/ 4u
What is the deployment process?                             5/ 4u     10/ 5u

(format: total_chunks / unique_sources)


---
### Strategy 5 — Similarity Search with Relevance Scores

Useful for understanding **how confident** the retriever is. Low scores
may indicate the query is out-of-domain or the top chunks are borderline relevant.

In [10]:
query = "How is state management handled in the application?"
print(f"Query: {query}\n")

results_with_scores = vectorstore.similarity_search_with_relevance_scores(query, k=5)

print(f"{'Score':<8} {'Source':<45} Section")
print("-" * 100)
for doc, score in results_with_scores:
    print(f"{score:.4f}  {doc.metadata['source']:<45} {doc.metadata['section'][:40]}")

print(f"\nScore range: {results_with_scores[-1][1]:.4f} – {results_with_scores[0][1]:.4f}")
print("(Higher = more relevant. Scores below ~0.3 are often noise.)")

Query: How is state management handled in the application?

Score    Source                                        Section
----------------------------------------------------------------------------------------------------
0.4071  architecture/overview.md                      When to Use What
0.3957  architecture/code-quality.md                  What Goes Where
0.2678  architecture/redux.md                         Overview
0.2566  features/journey-canvas.md                    Builder Slice (`builderSlice`)
0.2030  architecture/overview.md                      Overview

Score range: 0.2030 – 0.4071
(Higher = more relevant. Scores below ~0.3 are often noise.)


In [11]:
# Test with an out-of-domain query to see score drop-off

query = "What is the weather like in Mumbai?"
print(f"Query: {query}\n")

results_with_scores = vectorstore.similarity_search_with_relevance_scores(query, k=5)

print(f"{'Score':<8} {'Source':<45} Section")
print("-" * 100)
for doc, score in results_with_scores:
    print(f"{score:.4f}  {doc.metadata['source']:<45} {doc.metadata['section'][:40]}")

print(f"\n⚠️  Low scores confirm this query is out-of-domain.")
print("You can use a score threshold to avoid returning irrelevant results.")

Query: What is the weather like in Mumbai?

Score    Source                                        Section
----------------------------------------------------------------------------------------------------
-0.1481  architecture/code-quality.md                  Code Quality & Style Guide for Developer
-0.1897  features/action-builder.md                    Firestore Collections
-0.2050  architecture/theming.md                       Multi-Theming and Styling -- In-Depth Gu
-0.2102  architecture/overview.md                      Firestore Compatibility Layer
-0.2186  architecture/overview.md                      Environment Isolation

⚠️  Low scores confirm this query is out-of-domain.
You can use a score threshold to avoid returning irrelevant results.


/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_23455/4130558646.py:6: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b2725ceb-44da-4832-b76a-3fdd5ca2857c', metadata={'source': 'architecture/code-quality.md', 'doc_title': 'Code Quality & Style Guide for Developers', 'doc_type': 'architecture', 'audience': 'all', 'owner': '', 'status': '', 'version': '', 'last_modified': 'March 2026', 'section': 'Code Quality & Style Guide for Developers', 'heading_hierarchy': ['Code Quality & Style Guide for Developers'], 'chunk_index': 0}, page_content='# Code Quality & Style Guide for Developers  \n> **Parent:** [Architecture](./architecture.md)\n> **Last Updated:** March 2026  \n---'), np.float32(-0.14806843)), (Document(id='246ba10e-eaae-442b-ba37-37d913a2974d', metadata={'source': 'features/action-builder.md', 'doc_title': 'Action Builder — Technical Documentation', 'doc_type': 'feature', 'audience': 'all', 'owner': 'Admin Tool Engineering', 'status': 'product

---
### Takeaways

| Strategy | Recall | Precision | Latency | Best for |
|----------|--------|-----------|---------|----------|
| Baseline (k=5) | Moderate | Good | Fast | Simple queries |
| Metadata filter | Targeted | High | Fast | Scoped questions ("architecture only") |
| Multi-query | High | Moderate | Slower (1 LLM + N embed calls) | Ambiguous or broad questions |
| Score threshold | — | Higher | Same | Filtering out noise / out-of-domain |

**Recommended combination for the app:**
- Use **multi-query** as the default retriever for better recall
- Let the agent apply **metadata filters** when it can infer the doc_type from the question
- Use **relevance scores** to set a threshold and avoid hallucinating on out-of-domain queries